# brier - M7.2: how many L0 rotations are needed?

L0 asks a 20-option question once per rotation of the options (20 forward passes on
banking20). This notebook runs every rotation once and scores L0 with 1, 2, 4, 5, 10 and 20
evenly spaced rotations, on the same split and seed as the published results:

* **Qwen3-1.7B** (chat model)
* **Falcon3-1B-Base** (base model; its raw answer flips 99 % of the time when the options are
  reversed)

**Runtime:** A100 or L4, about 15 min per model. Results are saved to Google Drive after each
model (`SAVE_TO_DRIVE = True`), and a re-run skips models that already have results.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRIER_COMMIT = "e28a7eb0ba59c56869c500b6d5708d2909cf15cf"  # M7.2 rotations command
DTYPE = "bfloat16"
COUNTS = "1,2,4,5,10,20"
SAVE_TO_DRIVE = True
MODELS = {
    "qwen3": ("Qwen/Qwen3-1.7B", "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e", 64),
    "falcon3base": ("tiiuae/Falcon3-1B-Base", "cb37ef3559b157b5c9d9226296ba01a5162da1f7", 64),
}

In [ ]:
!rm -rf /content/brier /content/brier_repo
!git clone -q https://github.com/mohdUwaish59/brier.git /content/brier_repo
!git -C /content/brier_repo checkout -q {BRIER_COMMIT}
!pip install -q -e "/content/brier_repo[hf]"

In [ ]:
OUT_ROOT = "/content/results_m72"
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/brier_m72"
print("results go to", OUT_ROOT)

In [ ]:
import glob
import subprocess
import sys
import time

# brier runs in fresh processes only (this notebook's Python never imports it).
for key, (model_id, revision, batch) in MODELS.items():
    out = f"{OUT_ROOT}/{key}"
    if glob.glob(f"{out}/*_rotations.json"):
        print(f"== {key}: results exist, skipping")
        continue
    print(f"\n===== {key}: {model_id} @ {revision[:7]} =====")
    print("started", time.strftime("%H:%M:%S"), flush=True)
    cmd = [
        sys.executable,
        "-m",
        "brier.bench",
        "rotations",
        "--model",
        model_id,
        "--revision",
        revision,
        "--dtype",
        DTYPE,
        "--batch-size",
        str(batch),
        "--counts",
        COUNTS,
        "--out",
        out,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd="/content/brier_repo")  # noqa: S603
    print("finished", time.strftime("%H:%M:%S"))
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        print(f"!! {key} failed:\n{proc.stderr[-3000:]}")

In [ ]:
import glob
import json


def fmt(v):
    return f"{v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]"


for key in MODELS:
    files_ = glob.glob(f"{OUT_ROOT}/{key}/*_rotations.json")
    if not files_:
        print(f"{key}: no results")
        continue
    with open(files_[0]) as f:
        res = json.load(f)
    print(f"\n{key}: {res['model']['id']}  n_test={res['n_test']}  brier {res['git_commit']}")
    print("| rotations | accuracy | ece | nll | flip rate |")
    print("|---|---|---|---|---|")
    for m, e in sorted(res["rotation_curve"].items(), key=lambda kv: int(kv[0])):
        mt = e["metrics"]
        print(
            f"| {m} | {fmt(mt['accuracy'])} | {fmt(mt['ece'])} | {fmt(mt['nll'])} | {fmt(mt['flip_rate'])} |"
        )

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/brier_m72_results", "zip", OUT_ROOT)
files.download("/content/brier_m72_results.zip")